In [1]:
from mlp import Linear, BatchNorm1D, Tanh
import torch 
import torch.nn.functional as F
from contextensor import ContextTorchTensor, TensorSplit


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "/Users/vicentearjona/Documents/LLM_practice/.venv/lib/python3.9/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()

# Generate torch tensors from name list

In [2]:
list_to_tensor = ContextTorchTensor(context=4)
list_to_tensor.open(file='names.txt')
X, Y = list_to_tensor.get_tensors()


# Train/validation/test splits

In [ ]:
tsplit = TensorSplit(train_size=0.8 , test_size=0.1)
xtrain, xval, xtest, ytrain, yval, ytest = tsplit.split(xs=X, ys=Y)

# Initialize parameters 

In [4]:
dim_emb = 3 # embedding dimension
batch_size = 32 # number examples extracted for each batch. 
dim_hidden = 100 # dimensionality hidden layers. Number of neurons of the hidden layers 
g = 2147483647
vocab_size = list_to_tensor.vocab_size
context = list_to_tensor.context
num_iterations = 200000 
lr = 0.1

## Definition of the tensors

In [5]:
C = torch.randn(size=(vocab_size, dim_emb), generator=torch.Generator().manual_seed(g)) # embedding tensor 
# Definition of the layer elements 
layers = [Linear(fan_in=dim_emb * context, fan_out=dim_hidden,generator=g), Tanh(),
          Linear(fan_in=dim_hidden, fan_out=dim_hidden, generator=g),       Tanh(),
          Linear(fan_in=dim_hidden, fan_out=dim_hidden, generator=g),       Tanh(),
          Linear(fan_in=dim_hidden, fan_out=vocab_size, generator=g,bias=False)]    

## Solving initialization issues

In [6]:
with torch.no_grad():
    layers[-1].weight *= 0.01
    for layer in layers[:-1]:
        if isinstance(layer, Linear):
            layer.bias *= 0.01 
            layer.weight *= 5/3

## Parameters 

In [7]:
parameters = [C] + [p for layer in layers for p in layer.parameters()]
for p in parameters:
    p.requires_grad = True

In [8]:
print(f'number of parameters = {sum([p.nelement() for p in parameters])}')

number of parameters = 24281


# Forward and backward pass 

In [ ]:
# For loop to update the parameters of the NN via gradient descent 
loss_train = [] # object where we store the loss of each iteration 
up_to_data = [] # object where we store the update to date data of each iteration 
for i in range(num_iterations):
# Select minibatch from the complete set 
    ixs = torch.randint(low=0, high=xtrain.shape[0], size=(batch_size,))
    xtr = xtrain[ixs]
    ytr = ytrain[ixs]
# Forward pass. Returns loss
    emb = C[xtr]
    X = emb.view(-1, emb.shape[1] * emb.shape[2])
# Iteration over layers 
    for layer in layers:
        X = layer(X)
# Append loss 
    loss_i = F.cross_entropy(input=X, target=ytr)
    loss_train.append(loss_i.log10().item())
# Backward pass. Updates grads
    for layer in layers:
        layer.out.retain_grad() 
    for p in parameters: 
        p.grad = None 
    loss_i.backward() # fill grad attributes. Gradient descent for the loss 
# update pass. Recalculates params
    lr = lr if i < int(0.75 * num_iterations) else lr / 100
    for p in parameters:
        p.data += -lr * p.grad
        # Store up to date pass. For each iteration, we store the update to data of each parameter 
        with torch.no_grad():
            up_to_data.append(((lr * p.grad).std() / p.data.std()).log10().item())
    if i % 10000 == 0:
        print(f"Iteration:{i}")


Iteration:0


In [ ]:
with torch.no_grad(): 
    def split_loss(split:str): 
        X, Y = {
            'train': [xtrain, ytrain], 
            'test': [xtest, ytest], 
            'val': [xval, yval]
        }[split]
        emb = C[X]
        X = emb.view(-1, emb.shape[1] * emb.shape[2])
        # Iteration over layers 
        for layer in layers:
            X = layer(X)
        # Append loss 
        loss_i = F.cross_entropy(input=X, target=Y)



# Applying BATCHNORM after each linear layer 

## Definition of the tensors

In [13]:
C = torch.randn(size=(vocab_size, dim_emb), generator=torch.Generator().manual_seed(g)) # embedding tensor 
# Definition of the layer elements 
layers = [Linear(fan_in=dim_emb * context, fan_out=dim_hidden,generator=g), BatchNorm1D(dim_hidden), Tanh(),
          Linear(fan_in=dim_hidden, fan_out=dim_hidden, generator=g),       BatchNorm1D(dim_hidden), Tanh(),
          Linear(fan_in=dim_hidden, fan_out=dim_hidden, generator=g),       BatchNorm1D(dim_hidden), Tanh(),
          Linear(fan_in=dim_hidden, fan_out=vocab_size, generator=g),       BatchNorm1D(vocab_size)]    

## Solving initialization issues

In [14]:
with torch.no_grad():
    layers[-1].bngain *= 0.01
    for layer in layers[:-1]:
        if isinstance(layer, Linear):
            layer.bias *= 0.01 
            layer.weight *= 5/3

## Parameters 

In [15]:
parameters = [C] + [p for layer in layers for p in layer.parameters()]
for p in parameters:
    p.requires_grad = True

In [16]:
print(f'number of parameters = {sum([p.nelement() for p in parameters])}')

number of parameters = 24962


# Forward and backward pass 

In [17]:
# For loop to update the parameters of the NN via gradient descent 
loss_train = [] # object where we store the loss of each iteration 
up_to_data = [] # object where we store the update to date data of each iteration 
for i in range(num_iterations):
# Select minibatch from the complete set 
    ixs = torch.randint(low=0, high=xtrain.shape[0], size=(batch_size,))
    xtr = xtrain[ixs]
    ytr = ytrain[ixs]
# Forward pass. Returns loss
    emb = C[xtr]
    X = emb.view(-1, emb.shape[1] * emb.shape[2])
# Iteration over layers 
    for layer in layers:
        X = layer(X)
# Append loss 
    loss_i = F.cross_entropy(input=X, target=ytr)
    loss_train.append(loss_i.log10().item())
# Backward pass. Updates grads
    for layer in layers:
        layer.out.retain_grad() 
    for p in parameters: 
        p.grad = None 
    loss_i.backward() # fill grad attributes. Gradient descent for the loss 
# update pass. Recalculates params
    lr = lr if i < int(0.75 * num_iterations) else lr / 100
    for p in parameters:
        p.data += -lr * p.grad
        # Store up to date pass. For each iteration, we store the update to data of each parameter 
        with torch.no_grad():
            up_to_data.append(((lr * p.grad).std() / p.data.std()).log10().item())
    if i % 10000 == 0:
        print(f"Iteration:{i}")


Iteration:0
Iteration:10000
Iteration:20000
Iteration:30000
Iteration:40000
Iteration:50000
Iteration:60000
Iteration:70000
Iteration:80000
Iteration:90000
Iteration:100000
Iteration:110000
Iteration:120000
Iteration:130000
Iteration:140000
Iteration:150000
Iteration:160000
Iteration:170000
Iteration:180000
Iteration:190000
